In [ ]:
import pandas as pd
import numpy as np
import string
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import nltk

# Download necessary NLTK data files
nltk.download('wordnet')

# Preprocessing function without removing stopwords
def preprocess_text(text):
    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Lemmatize words
    lemmatizer = WordNetLemmatizer()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

# Load and prepare the training data
train_df = pd.read_csv("bugs-train.csv")
test_df = pd.read_csv("bugs-test.csv")
severity_mapping = {
    'enhancement': 1,
    'trivial': 2,
    'minor': 3,
    'normal': 4,
    'major': 5,
    'blocker': 6,
    'critical': 7
}
train_df['severity'] = train_df['severity'].map(severity_mapping)

# Apply preprocessing to the text data
train_df['summary'] = train_df['summary'].apply(preprocess_text)
test_df['summary'] = test_df['summary'].apply(preprocess_text)

# Prepare text data
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_df['summary'])
X_test_tfidf = vectorizer.transform(test_df['summary'])

# Train separate models for each class using stratified sampling
models = {}
for severity, label in severity_mapping.items():
    y_binary = train_df['severity'] == label
    X_train, X_val, y_train, y_val = train_test_split(X_train_tfidf, y_binary, test_size=0.1, random_state=42, stratify=y_binary)

    # Define AdaBoostClassifier
    model = AdaBoostClassifier()

    # Hyperparameter tuning using GridSearchCV
    param_grid = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0]
    }
    grid_search = GridSearchCV(model, param_grid, scoring='f1', cv=3, verbose=1)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    models[label] = best_model

    # Evaluate model on the validation set (optional, can be removed)
    y_pred = best_model.predict(X_val)
    print(f"Class {label} - Accuracy: {accuracy_score(y_val, y_pred)}, "
          f"Precision: {precision_score(y_val, y_pred, average='macro')}, "
          f"Recall: {recall_score(y_val, y_pred, average='macro')}, "
          f"F1 Score: {f1_score(y_val, y_pred, average='macro')}")

# Meta-learner: Train an SVM model using the predictions of AdaBoost models
def train_meta_learner(models, X, y):
    meta_features = np.column_stack([model.predict(X) for model in models.values()])
    meta_learner = SVC(kernel='linear', probability=True)

    # Normalize class labels for the meta-learner
    y_normalized = y - 1

    meta_learner.fit(meta_features, y_normalized)
    return meta_learner

# Prepare meta-features for the meta-learner
meta_features_train = np.column_stack([model.predict(X_train_tfidf) for model in models.values()])
meta_features_test = np.column_stack([model.predict(X_test_tfidf) for model in models.values()])

# Train the meta-learner
meta_learner = train_meta_learner(models, X_train_tfidf, train_df['severity'])

# Predict with meta-learner
predictions_normalized = meta_learner.predict(meta_features_test)
final_predictions = predictions_normalized + 1

# Evaluate the ensemble model
ensemble_accuracy = accuracy_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1)
ensemble_precision = precision_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')
ensemble_recall = recall_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')
ensemble_f1 = f1_score(train_df['severity'], meta_learner.predict(meta_features_train) + 1, average='macro')

print(f"Ensemble Model - Accuracy: {ensemble_accuracy}, "
      f"Precision: {ensemble_precision}, "
      f"Recall: {ensemble_recall}, "
      f"F1 Score: {ensemble_f1}")

# Map numeric labels back to string labels
reverse_severity_mapping = {v: k for k, v in severity_mapping.items()}
predicted_severities = [reverse_severity_mapping[label] for label in final_predictions]

# Create a submission DataFrame
submission_df = pd.DataFrame({
    'bug_id': test_df['bug_id'],
    'severity': predicted_severities  # Change column name to 'severity'
})

# Save to CSV
submission_path = "submission45.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved to {submission_path}")


[nltk_data] Downloading package wordnet to /root/nltk_data...


Fitting 3 folds for each of 9 candidates, totalling 27 fits
Class 1 - Accuracy: 0.9715, Precision: 0.6643963926320554, Recall: 0.5171266344358639, F1 Score: 0.5255515463273944
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Class 2 - Accuracy: 0.9923125, Precision: 0.6781906879161241, Recall: 0.5164462636439967, F1 Score: 0.5286045761285645
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Class 3 - Accuracy: 0.9805, Precision: 0.6987949295304812, Recall: 0.5078414441086371, F1 Score: 0.510603396992799
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Class 4 - Accuracy: 0.8586875, Precision: 0.8327457383505185, Recall: 0.7103811192004283, F1 Score: 0.746081614167377
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Class 5 - Accuracy: 0.9626875, Precision: 0.765422417398578, Recall: 0.5272547435439756, F1 Score: 0.541601730511055
Fitting 3 folds for each of 9 candidates, totalling 27 fits
Class 6 - Accuracy: 0.995125, Precision: 0.581206738387

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Ensemble Model - Accuracy: 0.8540731759146989, Precision: 0.39820477962807665, Recall: 0.2577737084703527, F1 Score: 0.25741560962319965
Submission file saved to submission45.csv
